# 🌟 Captura AI Service - Google Colab
Gunakan notebook ini untuk menjalankan dan melakukan testing pada AI Service secara interaktif di Google Colab.

In [ ]:
# 1. Clone repository
!git clone https://github.com/Captura-AI/artificial-intelligence.git
%cd artificial-intelligence/ai-service

# 2. Install Dependencies
!pip install -r requirements.txt


In [ ]:
# 3. Test AI Pipeline
import os
import urllib.request
from PIL import Image
import IPython.display as display

from app.config import get_settings
from app.services.vehicle_detector import detect_vehicle
from app.services.plate_reader import read_license_plate
from app.services.clip_embedder import classify_scene_tags

settings = get_settings()

# Gunakan sample image (ganti URL jika perlu)
IMAGE_URL = 'https://images.unsplash.com/photo-1568605117036-5fe5e7bab0b7?q=80&w=1000&auto=format&fit=crop'
urllib.request.urlretrieve(IMAGE_URL, 'sample.jpg')
image = Image.open('sample.jpg').convert('RGB')

print('📸 Menampilkan Gambar:')
display.display(image.resize((400, 300)))

print('\n🚗 1. Mendeteksi Kendaraan (YOLOv8)...')
vehicles = detect_vehicle(image, model_path=settings.yolo_model_path)
print(f'Ditemukan {len(vehicles)} kendaraan.')

print('\n🔠 2. Membaca Plat Nomor (EasyOCR)...')
for v_type, v_conf, bbox in vehicles:
    cropped = image.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
    plate, p_conf = read_license_plate(cropped, languages=settings.ocr_languages)
    print(f'- Kendaraan: {v_type} | Plat: {plate} (Confidence: {p_conf})')

print('\n🏷️ 3. Mengklasifikasi Scene Tags (CLIP)...')
tags = classify_scene_tags(image, model_name=settings.clip_model_name)
print(f'Tags: {tags}')
